In [17]:
from pydantic import BaseModel
from typing import Any, Optional
from unstructured.partition.pdf import partition_pdf
from langchain.vectorstores import Chroma
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.llms import OpenAI
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
import uuid
from langchain.schema import Document
from langchain.storage import InMemoryStore
from langchain.retrievers import MultiVectorRetriever
from pprint import pprint
from langchain.schema.runnable import RunnablePassthrough

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker

In [18]:

from dotenv import load_dotenv
import os
load_dotenv()

os.environ['HF_TOKEN']=os.getenv("HF_TOKEN")
HF_TOKEN=os.environ['HF_TOKEN']

rerank_model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")

In [19]:


# Load 
path = "../"
file = "protocol_ex1.pdf"

# Get elements
raw_pdf_elements = partition_pdf(
    filename=path+file,
    extract_images_in_pdf=False,
    infer_table_structure=True, 
    # Post processing to aggregate text once we have the title 
    chunking_strategy="by_title",
    # Chunking params to aggregate text blocks
    # Require maximum chunk size of 4000 chars
    # Attempt to create a new chunk at 3800 chars
    # Attempt to keep chunks > 2000 chars 
    max_characters=4000, 
    new_after_n_chars=3800, 
    combine_text_under_n_chars=2000,
    image_output_dir_path=path
)

In [42]:
class Element(BaseModel):
    type: str
    page_content: Any

# Categorize by type
categorized_elements = []
for element in raw_pdf_elements:
    print(f"{type(element)}, {dir(element)=}")
    if "unstructured.documents.elements.Table" in str(type(element)):
        categorized_elements.append(Element(type="table", page_content=str(element)))
    elif "unstructured.documents.elements.CompositeElement" in str(type(element)):
        categorized_elements.append(Element(type="text", page_content=str(element)))

# Tables
table_elements = [e for e in categorized_elements if e.type == "table"]
print(len(table_elements)) 
# output: 28 elements in the PDF file

# Text
text_elements = [e for e in categorized_elements if e.type == "text"]
print(len(text_elements)) 

<class 'unstructured.documents.elements.CompositeElement'>, dir(element)=['__abstractmethods__', '__annotations__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '__weakref__', '_abc_impl', '_element_id', 'apply', 'category', 'convert_coordinates_to_new_system', 'embeddings', 'id', 'id_to_hash', 'metadata', 'text', 'to_dict']
<class 'unstructured.documents.elements.CompositeElement'>, dir(element)=['__abstractmethods__', '__annotations__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '_

In [21]:
table_elements

[Element(type='table', page_content='PROTOCOL SUMMARY ........cc:ccccsscsccessssseecesseesescessesseeseeseesseesevsessaesaevsoesaaseesseesaevoeseasaas 10 BACKGROUND AND TRIAL RATIONALE ..........cccceecsecssesseeseeseesseensenseeseeseevenneaes 23 TRIAL OBJECTIVES AND ENDPOINTS. 2.1. Objectives .........ceceeeeeee 2.1.1. Primary Objective. 2.1.2. Secondary Objecti 29 2.2. Trial Endpoints ................ .29 2.2.1. Primary Endpoints 29 2.2.2. Secondary Endpoints ...... . 30 TRIAL DESIGN AND TRIAL DESIGN RATIONALE 3.1. Trial MOSIQM .0....eeeeeseeceeeeeeeeeeeeeeeeeeeeeeeeeseseaeeeeeeaseaeeeeesaesaaseeseaeeaaeeeeesessaseaeeaeseaseeeseeseasseeeeeseaseeeeeaseeaseees 30 3.2. Trial duration and duration of patient participation . 3.3. Rationale of trial design SELECTION OF PATIENTS 41. Screening Criteria... cece cee ceeceeeeeseeseseeceesessesessevsesassesaesoeseseeseusesadseusesausesaesasseseseseesaasegsese 37 4.2. Inclusion criteria 4.3. Exclusion criteria .... SCHEDULE OF EVENTS... ENROLMENT

In [22]:
text_elements

[Element(type='text', page_content="DNDi Drugs for Neglected Diseases initiative\n\nCLINICAL TRIAL PROTOCOL\n\nDouble-blind, Double-dummy, Phase 2 Randomized, Multicenter, Proof-of-Concept, Safety and\n\nEfficacy Trial to Evaluate Different Oral Benznidazole Monotherapy and Benznidazole/E1224 Combination Regimens for the Treatment of Adult Patients with Chronic Indeterminate Chagas Disease. Short title BENDITA BEnznidazole New Doses Improved Treatment and Associations Name of product(s) E1224 (Fosravuconazole drug substance equivalent to 100 mg of Ravuconazole), Abarax (Benznidazole; N-benzil-2-nitro-1- imidazolacetamide), and respective matched Placebos Drug Class Triazole and Nitro-imidazole Phase Investigational — Phase 2 trial Indication Chronic Indeterminate Chagas Disease Protocol Number DNDi-CH-E1224-003 EudraCT NA Sponsor DNDi, Chemin Louis Dunant, 15, 1202 GENEVA Switzerland Phone: +41 22 906 9230 Manufacturers Laboratdério Elea, Buenos Aires, Argentina Eisai Co, Ltd., Tokyo, 

In [23]:
# The vectorstore to use to index the child chunks
vectorstore = Chroma(
    collection_name="rag_with_summaries",
    embedding_function=OpenAIEmbeddings()
)

# The storage layer for the parent documents
store = InMemoryStore()
id_key = "doc_id"

# The retriever (empty to start)
retriever = MultiVectorRetriever(
    vectorstore=vectorstore, 
    docstore=store, 
    id_key=id_key,
)

# Add texts
doc_ids = [str(uuid.uuid4()) for _ in text_elements]
summary_texts = [Document(page_content=s.page_content,metadata={id_key: doc_ids[i]}) for i, s in enumerate(text_elements) if s.page_content]
retriever.vectorstore.add_documents(summary_texts)
retriever.docstore.mset(list(zip(doc_ids, text_elements)))

# Add tables
table_ids = [str(uuid.uuid4()) for _ in table_elements]
summary_tables = [Document(page_content=s.page_content,metadata={id_key: table_ids[i]}) for i, s in enumerate(table_elements) if s.page_content]
retriever.vectorstore.add_documents(summary_tables)
retriever.docstore.mset(list(zip(table_ids, table_elements)))


In [24]:

compressor = CrossEncoderReranker(model=rerank_model, top_n=2)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)
    

In [25]:
data = compression_retriever.invoke("screeing criteria")
pprint(data)

[Element(type='text', page_content='8.2. Screening and Baseline Assessments\n\nDuring screening, and after obtaining Informed Consent, the following assessments will be done in order to evaluate patient eligibility for the trial:\n\ne Complete medical history with an emphasis on CD;\n\nConfidential\n\nConfidential\n\nPage 50 of 87\n\nProtocol number DNDi-CH-E1224-003\n\nMay 04, 2018. Version 5.0.\n\nDemographic data and history of medications;\n\nPhysical examination, body weight and height and vital signs;\n\nChagas Disease serology: serum sample will be collected at screening for conventional CD serology. Patient must have at least 2 positive tests (among Conventional ELISA, Recombinant ELISA or IIF) to be eligible;\n\nReal-time PCR for qualitative and quantitative assessment (q-PCR) for all patients. In addition to positive serological tests, the patient must at least have one positive qualitative PCR test to be eligible for this trial;\n\nClinical safety laboratory evaluations: CBC

In [26]:
system_prompt = f"""
You are a medical document analysis assistant. 
Your task is to extract the Inclusion and Exclusion Criteria from clinical trial protocol documents. 
The criteria may include various conditions, including age ranges, diagnostic criteria, laboratory values, medical history, medication use, 
and other factors. You should identify the following key components:

    Inclusion Criteria: Extract conditions that participants must meet to be eligible for the trial. This includes but is not limited to:
        Age ranges (e.g., “18-65 years”).
        Medical conditions or diagnoses (e.g., “confirmed diagnosis of Type 2 Diabetes Mellitus”).
        Laboratory values or measurements (e.g., “HbA1c between 7.0% and 9.5%”).
        Time-based conditions (e.g., “diagnosed for at least 6 months”).
        Medication use criteria (e.g., “on stable doses of metformin for at least 3 months”).
        Consent requirements (e.g., “able to provide written informed consent”).

    Exclusion Criteria: Extract conditions that disqualify participants from the trial. This includes but is not limited to:
        Medical conditions (e.g., “history of cardiovascular disease”).
        Laboratory value thresholds (e.g., “serum creatinine > 1.5 mg/dL”).
        Medication use restrictions (e.g., “use of insulin therapy within the last 6 months”).
        Substance use (e.g., “history of drug abuse in the last 12 months”).
        Pregnancy or lactation conditions (e.g., “pregnant or breastfeeding women”).
        Other disqualifying conditions (e.g., “participation in another clinical trial in the last 3 months”).

Use the following structured format to organize your output:
Inclusion Criteria:
    [Condition 1]
    [Condition 2]
    [Condition 3]
    ...

Exclusion Criteria:
    [Condition 1]
    [Condition 2]
    [Condition 3]
    ...

Make sure to capture the key information from both Inclusion and Exclusion sections, including any numerical ranges, lab values,
 or conditions specified in the document.
 
Provide the data source information as well.

"""

In [37]:
# Prompt template
template = f"""Answer the question based only on the following context, which can include text and tables:
{{context}}
Question: {{question}}
"""
prompt = ChatPromptTemplate.from_template(system_prompt+template)

# LLM
model = ChatOpenAI(temperature=0,model="gpt-4")

# RAG pipeline
semi_structured_chain = (
    {"context": compression_retriever, "question": RunnablePassthrough()} 
    | prompt 
    | model 
    | StrOutputParser()
)

In [38]:
data = semi_structured_chain.invoke("which methods, lab tets/results are part of inclusion criteria?")
pprint(data)

('Inclusion Criteria:\n'
 '    1. Complete medical history with an emphasis on CD.\n'
 '    2. Demographic data and history of medications.\n'
 '    3. Physical examination, body weight and height, and vital signs.\n'
 '    4. Chagas Disease serology: patient must have at least 2 positive tests '
 '(among Conventional ELISA, Recombinant ELISA or IIF) to be eligible.\n'
 '    5. Real-time PCR for qualitative and quantitative assessment (q-PCR) for '
 'all patients. In addition to positive serological tests, the patient must at '
 'least have one positive qualitative PCR test to be eligible for this trial.\n'
 '    6. Clinical safety laboratory evaluations: CBC, ALT, AST, total and '
 'direct bilirubin, GGT, alkaline phosphatase, creatinine, fasting glucose, '
 'Ca, Mg and K will be assessed at screening and repeated at Day 0.\n'
 '    7. Morning serum cortisol: a morning blood sample will be taken for '
 'cortisol levels at screening visit.\n'
 '    8. The following parameters must be w

In [39]:
data = semi_structured_chain.invoke("what is the criteria for patient eligibility?")
pprint(data)

('Inclusion Criteria:\n'
 '    1. Women of reproductive potential must have a negative serum pregnancy '
 'test at screening.\n'
 '    2. EKG examination at screening must be normal for patient eligibility, '
 'i.e., PR ≤200 msec; QRS <120 msec; and QTc ≥ 350 msec and ≤450 msec interval '
 'durations in males, and QTc ≤470 msec in women.\n'
 '    3. Women of childbearing potential will undergo a second serum pregnancy '
 'test at treatment onset. Any positive test at baseline will automatically '
 'exclude the patient from this clinical trial.\n'
 '\n'
 'Exclusion Criteria:\n'
 '    1. Signs and/or symptoms of chronic cardiac and/or digestive form of '
 'CD.\n'
 '    2. History of cardiomyopathy, heart failure, or ventricular arrhythmia.\n'
 '    3. History of digestive surgery or mega syndromes.\n'
 '    4. Any other acute or chronic health conditions that may interfere with '
 'the efficacy and/or safety evaluation of the trial drug.\n'
 '    5. Laboratory test values considered clin

In [40]:
data = semi_structured_chain.invoke("list out the inclusion criteria and exclusion criteria.")
pprint(data)

('The document does not provide any Inclusion Criteria.\n'
 '\n'
 'Exclusion Criteria:\n'
 '1. Signs and/or symptoms of chronic cardiac and/or digestive form of CD.\n'
 '2. History of cardiomyopathy, heart failure, or ventricular arrhythmia.\n'
 '3. History of digestive surgery or mega syndromes.\n'
 '4. Any other acute or chronic health conditions that may interfere with the '
 'efficacy and/or safety evaluation of the trial drug (such as acute '
 'infections, history of HIV infection, diabetes, uncontrolled '
 'systolic/diastolic blood pressure, liver, and renal disease requiring '
 'medical treatment).\n'
 '5. Laboratory test values considered clinically significant or out of the '
 'allowable range at selection period as follows:\n'
 '    - Total WBC must be within the normal range, with an acceptable margin '
 'of +/- 5% (3,800 — 10,500/mm3).\n'
 '    - Platelets must be within the normal range up to 550,000/mm3.\n'
 '    - Total bilirubin must be within the normal range.\n'
 '   

In [31]:
# # Baseline
# vectorstore_baseline = Chroma.from_documents(
#     documents=all_splits,
#     collection_name="baseline_rag",
#     embedding=OpenAIEmbeddings()
# )

# retriever_baseline = vectorstore_baseline.as_retriever()